In [3]:
import os
import csv
import json
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

csv_file = "./filtered_urls/repairs_urls.csv"
output_folder = "./database/repairs_json_db"
os.makedirs(output_folder, exist_ok=True)

# Selenium options with better stealth
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize driver
driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

def scrape_product_page(url, soup):
    """Scrape product detail page"""
    product_data = {}
    product_data['url'] = url
    product_data['page_type'] = 'product'

    # Title
    title = soup.find('h1', class_='title-lg')
    product_data['title'] = title.text.strip() if title else None

    # Price
    price = soup.find('span', class_='js-partPrice')
    product_data['price'] = price.text.strip() if price else None

    # Availability
    availability = soup.find('span', itemprop='availability')
    product_data['availability'] = availability.get('content') if availability else None

    # Part Numbers
    ps_number = soup.find('span', class_='bold text-teal', itemprop='productID')
    product_data['partselect_number'] = ps_number.text.strip() if ps_number else None

    manufacturer_pn = soup.find('span', class_='bold text-teal', itemprop='mpn')
    product_data['manufacturer_part_number'] = manufacturer_pn.text.strip() if manufacturer_pn else None

    # Brand
    brand = soup.find('span', itemprop='brand')
    if brand:
        brand_name = brand.find('span', itemprop='name')
        product_data['brand'] = brand_name.text.strip() if brand_name else None
    else:
        product_data['brand'] = None

    # Description
    description = soup.find('div', itemprop='description')
    product_data['description'] = description.text.strip() if description else None

    # Image URLs
    images = []
    main_img = soup.find('img', itemprop='image')
    if main_img:
        img_src = main_img.get('src')
        if img_src:
            images.append(img_src)

    thumbs = soup.find_all('a', class_='js-part-img-thumb')
    for thumb in thumbs:
        img_url = thumb.get('data-large-src')
        if img_url and img_url not in images:
            images.append(img_url)

    product_data['images'] = images

    # Repair Rating
    repair_difficulty = soup.find('div', class_='pd__repair-rating__container__item__icon')
    if repair_difficulty:
        difficulty_text = repair_difficulty.find('p', class_='bold')
        product_data['repair_difficulty'] = difficulty_text.text.strip() if difficulty_text else None
    else:
        product_data['repair_difficulty'] = None

    # Repair Duration
    product_data['repair_duration'] = None
    duration_divs = soup.find_all('div', class_='d-flex')
    for div in duration_divs:
        text = div.text
        if 'mins' in text or 'minutes' in text.lower():
            product_data['repair_duration'] = text.strip()
            break

    # Replaces
    replaces_section = None
    for div in soup.find_all('div', class_='bold mb-1'):
        if 'replaces these:' in div.text:
            replaces_section = div
            break
    
    if replaces_section:
        replaces_div = replaces_section.find_next_sibling('div')
        product_data['replaces'] = replaces_div.text.strip() if replaces_div else None
    else:
        product_data['replaces'] = None

    # Symptoms
    symptoms = []
    symptoms_section = None
    for div in soup.find_all('div', class_='bold mb-1'):
        if 'fixes the following symptoms' in div.text:
            symptoms_section = div
            break
    
    if symptoms_section:
        symptom_list = symptoms_section.find_next('ul')
        if symptom_list:
            symptoms = [li.text.strip() for li in symptom_list.find_all('li')]
    product_data['fixes_symptoms'] = symptoms

    # Works with products
    product_types = []
    products_section = None
    for div in soup.find_all('div', class_='bold mb-1'):
        if 'works with the following products' in div.text:
            products_section = div
            break
    
    if products_section:
        product_list = products_section.find_next('ul')
        if product_list:
            product_types = [li.text.strip() for li in product_list.find_all('li')]
    product_data['works_with_products'] = product_types

    # Related Parts
    related_parts = []
    related_section = soup.find_all('div', class_='pd__related-part')
    for part in related_section:
        part_name = part.find('a', class_='bold')
        part_price = part.find('div', class_='title-md bold mt-2')
        if part_name and part_price:
            related_parts.append({
                'name': part_name.text.strip(),
                'price': part_price.text.strip().replace('$', '').strip(),
                'url': 'https://www.partselect.com' + part_name.get('href') if part_name.get('href') else None
            })
    product_data['related_parts'] = related_parts

    # Compatible Models
    compatible_models = []
    crossref_section = soup.find('div', class_='pd__crossref__list')
    if crossref_section:
        model_rows = crossref_section.find_all('div', class_='row', recursive=False)
        for row in model_rows[:20]:
            cols = row.find_all(['div', 'a'], recursive=False)
            if len(cols) >= 2:
                brand_col = cols[0]
                model_col = cols[1]
                
                brand = brand_col.text.strip() if brand_col else None
                model = model_col.text.strip() if model_col else None
                
                if brand and model:
                    compatible_models.append({
                        'brand': brand,
                        'model': model
                    })
    product_data['compatible_models'] = compatible_models

    # Metadata
    meta_data = soup.find('div', id='main')
    if meta_data:
        product_data['inventory_id'] = meta_data.get('data-inventory-id')
        product_data['category'] = meta_data.get('data-category')
        product_data['model_type'] = meta_data.get('data-modeltype')
    else:
        product_data['inventory_id'] = None
        product_data['category'] = None
        product_data['model_type'] = None

    return product_data

def scrape_repair_page(url, soup):
    """Scrape repair/troubleshooting page - ONLY repair fields"""
    repair_data = {}
    repair_data['url'] = url
    repair_data['page_type'] = 'repair'

    # Page title (e.g., "How To Fix A Noisy GE Dishwasher")
    title = soup.find('h1', class_='title-main')
    repair_data['page_title'] = title.text.strip() if title else None

    # Breadcrumbs (e.g., "REPAIR > GE DISHWASHER > NOISY")
    crumbs = soup.find('div', class_='crumbs')
    repair_data['breadcrumb'] = crumbs.text.strip() if crumbs else None

    # Extract structured info from breadcrumb
    repair_data['brand'] = None
    repair_data['appliance_type'] = None
    repair_data['symptom'] = None
    
    if crumbs:
        crumb_links = crumbs.find_all('a')
        if len(crumb_links) >= 1:
            # Last link is usually "BRAND APPLIANCE" like "GE DISHWASHER"
            last_link_text = crumb_links[-1].text.strip()
            parts = last_link_text.split()
            if len(parts) >= 2:
                repair_data['brand'] = parts[0]
                repair_data['appliance_type'] = ' '.join(parts[1:])
        
        # Symptom is after the last ">"
        full_text = crumbs.text.strip()
        if '>' in full_text:
            symptom = full_text.split('>')[-1].strip()
            repair_data['symptom'] = symptom

    # Main video tutorial
    video = soup.find('div', class_='yt-video')
    if video:
        video_id = video.get('data-yt-init')
        repair_data['main_video_id'] = video_id
        repair_data['main_video_url'] = f"https://www.youtube.com/watch?v={video_id}" if video_id else None
        
        video_thumb = video.find('img')
        if video_thumb:
            repair_data['main_video_thumbnail'] = video_thumb.get('data-src')
            repair_data['main_video_title'] = video_thumb.get('title') or video_thumb.get('alt')
    else:
        repair_data['main_video_id'] = None
        repair_data['main_video_url'] = None
        repair_data['main_video_thumbnail'] = None
        repair_data['main_video_title'] = None

    # About this repair stats
    intro = soup.find('div', class_='repair__intro')
    repair_data['difficulty_rating'] = None
    repair_data['total_repair_stories'] = None
    repair_data['total_videos'] = None
    
    if intro:
        stats_list = intro.find('ul', class_='list-disc')
        if stats_list:
            for li in stats_list.find_all('li'):
                text = li.text.strip()
                if 'EASY' in text.upper() or 'DIFFICULT' in text.upper():
                    repair_data['difficulty_rating'] = text
                elif 'repair stories' in text.lower():
                    # Extract number from "989 repair stories"
                    repair_data['total_repair_stories'] = text
                elif 'video' in text.lower():
                    # Extract from "10 step by step videos"
                    repair_data['total_videos'] = text

    # Quick navigation links (parts that can cause this symptom)
    quick_nav_links = []
    intro_links = intro.find_all('a', class_='js-scrollTrigger') if intro else []
    for link in intro_links:
        quick_nav_links.append({
            'part_name': link.text.strip(),
            'anchor_id': link.get('href', '').replace('#', '')
        })
    repair_data['quick_navigation'] = quick_nav_links

    # Common parts that cause this symptom (the main content)
    common_parts = []
    
    # Find all section titles (h2 with section-title class)
    section_titles = soup.find_all('h2', class_='section-title')
    
    for section_title in section_titles:
        part_info = {}
        
        # Part name and anchor ID
        part_info['part_name'] = section_title.text.strip()
        part_info['section_id'] = section_title.get('id')
        
        # Find the symptom-list__desc div that follows this title
        desc_section = section_title.find_next('div', class_='symptom-list__desc')
        
        if desc_section:
            # Get the left column with troubleshooting info
            left_col = desc_section.find('div', class_='col-lg-6')
            
            if left_col:
                # Extract description paragraphs
                paragraphs = []
                for p in left_col.find_all('p', recursive=False):
                    paragraphs.append(p.text.strip())
                part_info['description'] = paragraphs
                
                # Extract troubleshooting steps (usually in <ol>)
                steps = []
                step_list = left_col.find('ol')
                if step_list:
                    for li in step_list.find_all('li'):
                        steps.append(li.text.strip())
                part_info['troubleshooting_steps'] = steps
            
            # Get the right column with part category info
            right_col = desc_section.find_all('div', class_='col-lg-6')
            if len(right_col) > 1:
                right_col = right_col[-1]
            elif right_col:
                right_col = right_col[0]
            else:
                right_col = None
            
            if right_col:
                # Part category link
                category_link = right_col.find('a', href=lambda x: x and x.startswith('/'))
                if category_link:
                    part_info['parts_category_url'] = 'https://www.partselect.com' + category_link.get('href')
                    part_info['parts_category_text'] = category_link.text.strip()
                
                # Part thumbnail images
                thumbs_container = right_col.find('div', class_='js-thumbs')
                if thumbs_container:
                    thumbs = thumbs_container.find_all('img', class_='thumb')
                    part_info['part_example_images'] = [img.get('data-src') for img in thumbs if img.get('data-src')]
                else:
                    part_info['part_example_images'] = []
        
        common_parts.append(part_info)
    
    repair_data['common_failing_parts'] = common_parts

    # Page metadata
    meta_data = soup.find('div', id='main')
    if meta_data:
        repair_data['data_modeltype'] = meta_data.get('data-modeltype')
        repair_data['data_page_type'] = meta_data.get('data-page-type')
        repair_data['data_page_name'] = meta_data.get('data-page-name')
    
    return repair_data

# Load URLs
try:
    with open(csv_file, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        product_urls = [row['url'] for row in reader]
except FileNotFoundError:
    print(f"Error: Could not find {csv_file}")
    driver.quit()
    exit(1)

print(f"Found {len(product_urls)} URLs to scrape")

for idx, url in enumerate(product_urls, 1):
    try:
        print(f"\n[{idx}/{len(product_urls)}] Processing: {url}")
        
        driver.get(url)
        
        # Wait for page to load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "h1"))
            )
        except:
            print(f"  ⚠ Timeout waiting for page load")
        
        # Random delay
        time.sleep(random.uniform(2, 4))
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Determine page type and scrape accordingly
        if '/Repair/' in url:
            data = scrape_repair_page(url, soup)
            page_type = 'repair'
        else:
            data = scrape_product_page(url, soup)
            page_type = 'product'

        # Generate filename
        if page_type == 'product':
            if data.get('partselect_number'):
                filename = data['partselect_number']
            elif data.get('manufacturer_part_number'):
                filename = data['manufacturer_part_number']
            else:
                filename = url.split('/')[-1].replace('.htm', '')
        else:
            # For repair pages: brand-appliance-symptom
            parts = []
            if data.get('brand'):
                parts.append(data['brand'])
            if data.get('appliance_type'):
                parts.append(data['appliance_type'].replace(' ', '-'))
            if data.get('symptom'):
                parts.append(data['symptom'])
            filename = '-'.join(parts) if parts else url.split('/')[-2]
        
        # Clean filename
        filename = filename.replace("/", "_").replace(" ", "_").replace(":", "_")
        filename = filename + ".json"
        filepath = os.path.join(output_folder, filename)

        # Save to JSON
        with open(filepath, 'w', encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        print(f"  ✓ Saved {filename} ({page_type})")

    except Exception as e:
        print(f"  ✗ Failed for {url}: {str(e)}")
        continue
    
    # Random delay between requests
    time.sleep(random.uniform(1, 3))

driver.quit()
print("\n✓ All done! Scraped data saved to:", output_folder)

Found 123 URLs to scrape

[1/123] Processing: https://www.partselect.com/Repair/Dishwasher/Amana/
  ✓ Saved AMANA_DISHWASHER.json (repair)

[2/123] Processing: https://www.partselect.com/Repair/Dishwasher/Amana/Door-Latch-Failure/
  ✓ Saved AMANA-DISHWASHER-BROKEN_DOOR_LATCH.json (repair)

[3/123] Processing: https://www.partselect.com/Repair/Dishwasher/Amana/Leaking/
  ✓ Saved AMANA-DISHWASHER-LEAKING.json (repair)

[4/123] Processing: https://www.partselect.com/Repair/Dishwasher/Amana/Noisy/
  ✓ Saved AMANA-DISHWASHER-NOISY.json (repair)

[5/123] Processing: https://www.partselect.com/Repair/Dishwasher/Amana/Not-Cleaning-Properly/
  ✓ Saved AMANA-DISHWASHER-NOT_CLEANING_DISHES_PROPERLY.json (repair)

[6/123] Processing: https://www.partselect.com/Repair/Dishwasher/Amana/Not-Draining/
  ✓ Saved AMANA-DISHWASHER-NOT_DRAINING.json (repair)

[7/123] Processing: https://www.partselect.com/Repair/Dishwasher/Amana/Not-Drying-Properly/
  ✓ Saved AMANA-DISHWASHER-NOT_DRYING_PROPERLY.json (rep


[57/123] Processing: https://www.partselect.com/Repair/Dishwasher/Kitchenaid/Not-Draining/
  ✓ Saved KITCHENAID-DISHWASHER-NOT_DRAINING.json (repair)

[58/123] Processing: https://www.partselect.com/Repair/Dishwasher/Kitchenaid/Not-Drying-Properly/
  ✓ Saved KITCHENAID-DISHWASHER-NOT_DRYING_PROPERLY.json (repair)

[59/123] Processing: https://www.partselect.com/Repair/Dishwasher/Kitchenaid/Will-Not-Dispense-Detergent/
  ✓ Saved KITCHENAID-DISHWASHER-WON'T_DISPENSE_DETERGENT.json (repair)

[60/123] Processing: https://www.partselect.com/Repair/Dishwasher/Kitchenaid/Will-Not-Fill-Water/
  ✓ Saved KITCHENAID-DISHWASHER-WILL_NOT_FILL_WITH_WATER.json (repair)

[61/123] Processing: https://www.partselect.com/Repair/Dishwasher/Kitchenaid/Will-Not-Start/
  ✓ Saved KITCHENAID-DISHWASHER-WILL_NOT_START.json (repair)

[62/123] Processing: https://www.partselect.com/Repair/Dishwasher/Leaking/
  ✓ Saved LEAKING.json (repair)

[63/123] Processing: https://www.partselect.com/Repair/Dishwasher/LG/
  

  ✓ Saved LEAKING.json (repair)

[114/123] Processing: https://www.partselect.com/Repair/Refrigerator/Light-Not-Working/
  ✓ Saved LIGHT_WON'T_TURN_ON.json (repair)

[115/123] Processing: https://www.partselect.com/Repair/Refrigerator/Noisy/
  ✓ Saved TOO_NOISY.json (repair)

[116/123] Processing: https://www.partselect.com/Repair/Refrigerator/Not-Dispensing-Ice/
  ✓ Saved NOT_DISPENSING_ICE.json (repair)

[117/123] Processing: https://www.partselect.com/Repair/Refrigerator/Not-Dispensing-Water/
  ✓ Saved DISPENSER_WILL_NOT_DISPENSE_WATER.json (repair)

[118/123] Processing: https://www.partselect.com/Repair/Refrigerator/Not-Making-Ice/
  ✓ Saved WILL_NOT_MAKE_ICE.json (repair)

[119/123] Processing: https://www.partselect.com/Repair/Refrigerator/Refrigerator-Freezer-Too-Warm/
  ✓ Saved FRIDGE_FREEZER_TOO_WARM.json (repair)

[120/123] Processing: https://www.partselect.com/Repair/Refrigerator/Refrigerator-Too-Cold/
  ✓ Saved FRIDGE_TOO_COLD.json (repair)

[121/123] Processing: https://

In [4]:
import os
import csv
import json
import time
import random
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.common.by import By
from bs4 import BeautifulSoup

csv_file = "./filtered_urls/repairs_urls.csv"
output_folder = "./database"
os.makedirs(output_folder, exist_ok=True)

# Selenium options with better stealth
options = Options()
options.add_argument("--headless=new")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

# Initialize driver
driver = webdriver.Chrome(options=options)
driver.execute_script("Object.defineProperty(navigator, 'webdriver', {get: () => undefined})")

def scrape_product_page(url, soup):
    """Scrape product detail page"""
    product_data = {}
    product_data['url'] = url
    product_data['page_type'] = 'product'

    # Title
    title = soup.find('h1', class_='title-lg')
    product_data['title'] = title.text.strip() if title else None

    # Price
    price = soup.find('span', class_='js-partPrice')
    product_data['price'] = price.text.strip() if price else None

    # Availability
    availability = soup.find('span', itemprop='availability')
    product_data['availability'] = availability.get('content') if availability else None

    # Part Numbers
    ps_number = soup.find('span', class_='bold text-teal', itemprop='productID')
    product_data['partselect_number'] = ps_number.text.strip() if ps_number else None

    manufacturer_pn = soup.find('span', class_='bold text-teal', itemprop='mpn')
    product_data['manufacturer_part_number'] = manufacturer_pn.text.strip() if manufacturer_pn else None

    # Brand
    brand = soup.find('span', itemprop='brand')
    if brand:
        brand_name = brand.find('span', itemprop='name')
        product_data['brand'] = brand_name.text.strip() if brand_name else None
    else:
        product_data['brand'] = None

    # Description
    description = soup.find('div', itemprop='description')
    product_data['description'] = description.text.strip() if description else None

    # Image URLs
    images = []
    main_img = soup.find('img', itemprop='image')
    if main_img:
        img_src = main_img.get('src')
        if img_src:
            images.append(img_src)

    thumbs = soup.find_all('a', class_='js-part-img-thumb')
    for thumb in thumbs:
        img_url = thumb.get('data-large-src')
        if img_url and img_url not in images:
            images.append(img_url)

    product_data['images'] = images

    # Repair Rating
    repair_difficulty = soup.find('div', class_='pd__repair-rating__container__item__icon')
    if repair_difficulty:
        difficulty_text = repair_difficulty.find('p', class_='bold')
        product_data['repair_difficulty'] = difficulty_text.text.strip() if difficulty_text else None
    else:
        product_data['repair_difficulty'] = None

    # Repair Duration
    product_data['repair_duration'] = None
    duration_divs = soup.find_all('div', class_='d-flex')
    for div in duration_divs:
        text = div.text
        if 'mins' in text or 'minutes' in text.lower():
            product_data['repair_duration'] = text.strip()
            break

    # Replaces
    replaces_section = None
    for div in soup.find_all('div', class_='bold mb-1'):
        if 'replaces these:' in div.text:
            replaces_section = div
            break
    
    if replaces_section:
        replaces_div = replaces_section.find_next_sibling('div')
        product_data['replaces'] = replaces_div.text.strip() if replaces_div else None
    else:
        product_data['replaces'] = None

    # Symptoms
    symptoms = []
    symptoms_section = None
    for div in soup.find_all('div', class_='bold mb-1'):
        if 'fixes the following symptoms' in div.text:
            symptoms_section = div
            break
    
    if symptoms_section:
        symptom_list = symptoms_section.find_next('ul')
        if symptom_list:
            symptoms = [li.text.strip() for li in symptom_list.find_all('li')]
    product_data['fixes_symptoms'] = symptoms

    # Works with products
    product_types = []
    products_section = None
    for div in soup.find_all('div', class_='bold mb-1'):
        if 'works with the following products' in div.text:
            products_section = div
            break
    
    if products_section:
        product_list = products_section.find_next('ul')
        if product_list:
            product_types = [li.text.strip() for li in product_list.find_all('li')]
    product_data['works_with_products'] = product_types

    # Related Parts
    related_parts = []
    related_section = soup.find_all('div', class_='pd__related-part')
    for part in related_section:
        part_name = part.find('a', class_='bold')
        part_price = part.find('div', class_='title-md bold mt-2')
        if part_name and part_price:
            related_parts.append({
                'name': part_name.text.strip(),
                'price': part_price.text.strip().replace('$', '').strip(),
                'url': 'https://www.partselect.com' + part_name.get('href') if part_name.get('href') else None
            })
    product_data['related_parts'] = related_parts

    # Compatible Models
    compatible_models = []
    crossref_section = soup.find('div', class_='pd__crossref__list')
    if crossref_section:
        model_rows = crossref_section.find_all('div', class_='row', recursive=False)
        for row in model_rows[:20]:
            cols = row.find_all(['div', 'a'], recursive=False)
            if len(cols) >= 2:
                brand_col = cols[0]
                model_col = cols[1]
                
                brand = brand_col.text.strip() if brand_col else None
                model = model_col.text.strip() if model_col else None
                
                if brand and model:
                    compatible_models.append({
                        'brand': brand,
                        'model': model
                    })
    product_data['compatible_models'] = compatible_models

    # Metadata
    meta_data = soup.find('div', id='main')
    if meta_data:
        product_data['inventory_id'] = meta_data.get('data-inventory-id')
        product_data['category'] = meta_data.get('data-category')
        product_data['model_type'] = meta_data.get('data-modeltype')
    else:
        product_data['inventory_id'] = None
        product_data['category'] = None
        product_data['model_type'] = None

    return product_data

def scrape_repair_page(url, soup):
    """Scrape repair/troubleshooting page - ONLY repair fields"""
    repair_data = {}
    repair_data['url'] = url
    repair_data['page_type'] = 'repair'

    # Page title (e.g., "How To Fix A Noisy GE Dishwasher")
    title = soup.find('h1', class_='title-main')
    repair_data['page_title'] = title.text.strip() if title else None

    # Breadcrumbs (e.g., "REPAIR > GE DISHWASHER > NOISY")
    crumbs = soup.find('div', class_='crumbs')
    repair_data['breadcrumb'] = crumbs.text.strip() if crumbs else None

    # Extract structured info from breadcrumb
    repair_data['brand'] = None
    repair_data['appliance_type'] = None
    repair_data['symptom'] = None
    
    if crumbs:
        crumb_links = crumbs.find_all('a')
        if len(crumb_links) >= 1:
            # Last link is usually "BRAND APPLIANCE" like "GE DISHWASHER"
            last_link_text = crumb_links[-1].text.strip()
            parts = last_link_text.split()
            if len(parts) >= 2:
                repair_data['brand'] = parts[0]
                repair_data['appliance_type'] = ' '.join(parts[1:])
        
        # Symptom is after the last ">"
        full_text = crumbs.text.strip()
        if '>' in full_text:
            symptom = full_text.split('>')[-1].strip()
            repair_data['symptom'] = symptom

    # Main video tutorial
    video = soup.find('div', class_='yt-video')
    if video:
        video_id = video.get('data-yt-init')
        repair_data['main_video_id'] = video_id
        repair_data['main_video_url'] = f"https://www.youtube.com/watch?v={video_id}" if video_id else None
        
        video_thumb = video.find('img')
        if video_thumb:
            repair_data['main_video_thumbnail'] = video_thumb.get('data-src')
            repair_data['main_video_title'] = video_thumb.get('title') or video_thumb.get('alt')
    else:
        repair_data['main_video_id'] = None
        repair_data['main_video_url'] = None
        repair_data['main_video_thumbnail'] = None
        repair_data['main_video_title'] = None

    # About this repair stats
    intro = soup.find('div', class_='repair__intro')
    repair_data['difficulty_rating'] = None
    repair_data['total_repair_stories'] = None
    repair_data['total_videos'] = None
    
    if intro:
        stats_list = intro.find('ul', class_='list-disc')
        if stats_list:
            for li in stats_list.find_all('li'):
                text = li.text.strip()
                if 'EASY' in text.upper() or 'DIFFICULT' in text.upper():
                    repair_data['difficulty_rating'] = text
                elif 'repair stories' in text.lower():
                    # Extract number from "989 repair stories"
                    repair_data['total_repair_stories'] = text
                elif 'video' in text.lower():
                    # Extract from "10 step by step videos"
                    repair_data['total_videos'] = text

    # Quick navigation links (parts that can cause this symptom)
    quick_nav_links = []
    intro_links = intro.find_all('a', class_='js-scrollTrigger') if intro else []
    for link in intro_links:
        quick_nav_links.append({
            'part_name': link.text.strip(),
            'anchor_id': link.get('href', '').replace('#', '')
        })
    repair_data['quick_navigation'] = quick_nav_links

    # Common parts that cause this symptom (the main content)
    common_parts = []
    
    # Find all section titles (h2 with section-title class)
    section_titles = soup.find_all('h2', class_='section-title')
    
    for section_title in section_titles:
        part_info = {}
        
        # Part name and anchor ID
        part_info['part_name'] = section_title.text.strip()
        part_info['section_id'] = section_title.get('id')
        
        # Find the symptom-list__desc div that follows this title
        desc_section = section_title.find_next('div', class_='symptom-list__desc')
        
        if desc_section:
            # Get the left column with troubleshooting info
            left_col = desc_section.find('div', class_='col-lg-6')
            
            if left_col:
                # Extract description paragraphs
                paragraphs = []
                for p in left_col.find_all('p', recursive=False):
                    paragraphs.append(p.text.strip())
                part_info['description'] = paragraphs
                
                # Extract troubleshooting steps (usually in <ol>)
                steps = []
                step_list = left_col.find('ol')
                if step_list:
                    for li in step_list.find_all('li'):
                        steps.append(li.text.strip())
                part_info['troubleshooting_steps'] = steps
            
            # Get the right column with part category info
            right_col = desc_section.find_all('div', class_='col-lg-6')
            if len(right_col) > 1:
                right_col = right_col[-1]
            elif right_col:
                right_col = right_col[0]
            else:
                right_col = None
            
            if right_col:
                # Part category link
                category_link = right_col.find('a', href=lambda x: x and x.startswith('/'))
                if category_link:
                    part_info['parts_category_url'] = 'https://www.partselect.com' + category_link.get('href')
                    part_info['parts_category_text'] = category_link.text.strip()
                
                # Part thumbnail images
                thumbs_container = right_col.find('div', class_='js-thumbs')
                if thumbs_container:
                    thumbs = thumbs_container.find_all('img', class_='thumb')
                    part_info['part_example_images'] = [img.get('data-src') for img in thumbs if img.get('data-src')]
                else:
                    part_info['part_example_images'] = []
        
        common_parts.append(part_info)
    
    repair_data['common_failing_parts'] = common_parts

    # Page metadata
    meta_data = soup.find('div', id='main')
    if meta_data:
        repair_data['data_modeltype'] = meta_data.get('data-modeltype')
        repair_data['data_page_type'] = meta_data.get('data-page-type')
        repair_data['data_page_name'] = meta_data.get('data-page-name')
    
    return repair_data

# Load URLs
try:
    with open(csv_file, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        product_urls = ['https://www.partselect.com/Repair/Dishwasher/Leaking/','https://www.partselect.com/Repair/Dishwasher/Will-Not-Start/']
except FileNotFoundError:
    print(f"Error: Could not find {csv_file}")
    driver.quit()
    exit(1)

print(f"Found {len(product_urls)} URLs to scrape")

for idx, url in enumerate(product_urls, 1):
    try:
        print(f"\n[{idx}/{len(product_urls)}] Processing: {url}")
        
        driver.get(url)
        
        # Wait for page to load
        try:
            WebDriverWait(driver, 10).until(
                EC.presence_of_element_located((By.TAG_NAME, "h1"))
            )
        except:
            print(f"  ⚠ Timeout waiting for page load")
        
        # Random delay
        time.sleep(random.uniform(2, 4))
        
        soup = BeautifulSoup(driver.page_source, 'html.parser')

        # Determine page type and scrape accordingly
        if '/Repair/' in url:
            data = scrape_repair_page(url, soup)
            page_type = 'repair'
        else:
            data = scrape_product_page(url, soup)
            page_type = 'product'

        # Generate filename
        if page_type == 'product':
            if data.get('partselect_number'):
                filename = data['partselect_number']
            elif data.get('manufacturer_part_number'):
                filename = data['manufacturer_part_number']
            else:
                filename = url.split('/')[-1].replace('.htm', '')
        else:
            # For repair pages: brand-appliance-symptom
            parts = []
            if data.get('brand'):
                parts.append(data['brand'])
            if data.get('appliance_type'):
                parts.append(data['appliance_type'].replace(' ', '-'))
            if data.get('symptom'):
                parts.append(data['symptom'])
            filename = '-'.join(parts) if parts else url.split('/')[-2]
        
        # Clean filename
        filename = filename.replace("/", "_").replace(" ", "_").replace(":", "_")
        filename = filename + ".json"
        filepath = os.path.join(output_folder, filename)

        # Save to JSON
        with open(filepath, 'w', encoding="utf-8") as f:
            json.dump(data, f, indent=2, ensure_ascii=False)

        print(f"  ✓ Saved {filename} ({page_type})")

    except Exception as e:
        print(f"  ✗ Failed for {url}: {str(e)}")
        continue
    
    # Random delay between requests
    time.sleep(random.uniform(1, 3))

driver.quit()
print("\n✓ All done! Scraped data saved to:", output_folder)

Found 2 URLs to scrape

[1/2] Processing: https://www.partselect.com/Repair/Dishwasher/Leaking/
  ✓ Saved LEAKING.json (repair)

[2/2] Processing: https://www.partselect.com/Repair/Dishwasher/Will-Not-Start/
  ✓ Saved WILL_NOT_START.json (repair)

✓ All done! Scraped data saved to: ./database
